In [ ]:
import AA500
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

# 5-21-25 Nutrient Analyzer Run QAQC

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250331 Run/'

result_05212025 = AA500.AA500_Result(result_dir + 'RCEW_NOx-PO4-NH4_Apr2025_2_2.csv', result_dir+'samplelist_20250331_2.xlsx')

## Drift and Baseline

In [ ]:
result_05212025.result_df

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_05212025.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_05212025.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_05212025.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- drift and baseline all look good except last ammonium drift

## Blank Check

In [ ]:
result_05212025.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- interesting, one DI Blank was pretty bad. contamination or off by one sample?

In [ ]:
result_05212025.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

In [ ]:
result_05212025.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- reagent blanks look good for nitrate and phosphate. 
- ammonium mean is high, but is pretty similar to DI blanks, so this seems like a problem with ammonium standards or milliQ is contaminated

## AA500 In-Sample Variation

In [ ]:
result_05212025.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of 1.7 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples ok, but a few bad ones - throw those out
- qa threshold of 5 ug/l

phosphate:
- also great- mean of 3 ug/L in range of 4 ug/L in SEAL datasheet
- also a bad one or two
- qa threshold of 4 ug/l

ammonium:
- way better than seal spec! 75% less than 1 ug/L, seal spec is 2 ug/L
- one bad one with a max of 7 ug/L
- qa threshold of 2 ug/l


In [ ]:
result_05212025.result_df.loc['RME']

In [ ]:
result_05212025.result_df.loc['Dobson']

- nice. for nitrate QA flag, RME has only one, Dobson has zero. 
- most flags for phosphate, a couple for ammonium. but no wholesale bad data

## Between Bottle Variation

In [ ]:
def plot_between_bottle_bar(result_df, analyte, **kwargs):
    # Reset index to ensure 'Sample Datetime' and 'Bottle Replicate' are columns
    df = result_df.reset_index()

    # Pivot so each bottle replicate is a column, indexed by date
    pivoted = df.pivot_table(
    index='Sample Datetime',
    columns='Bottle Replicate',
    values=analyte + ' mean'
    )

    # Optionally, get errors for yerr
    yerr = df.pivot_table(
        index='Sample Datetime',
        columns='Bottle Replicate',
        values=analyte + ' err'
    )

    # Plot
    ax = pivoted.plot(kind='bar', yerr=yerr, figsize=(10,6), rot=45, **kwargs)
    ax.set_ylabel(analyte+ ' mean')
    #ax.set_title(analyte + ' by Date and Bottle Replicate')
    ax.legend(title='Bottle Replicate')
    plt.tight_layout()
    plt.show()
    #return ax

plot_between_bottle_bar(result_05212025.result_df.loc['RME'], 'Nitrate')

#fig, ax=  plt.subplots()
#first_rep = result_05212025.result_df['Bottle Replicate']==1

#result_05212025.result_df.loc['RME'].groupby('Bottle Replicate').plot(y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='bar', rot=45)


- looks pretty good! good matches betweenn bottles except the 24th
- AA500 variation looks pretty good too, only the 2/25 sample is pretty bad
- these are also samples that i filtered... 

In [ ]:
plot_between_bottle_bar(result_05212025.result_df.loc['Dobson'], 'Nitrate')

#fig, ax=  plt.subplots(result_05212025.loc['Dobson'])
#first_rep = result_05212025.result_df['Bottle Replicate']==1

#result_05212025.result_df[first_rep].loc['Dobson'].reset_index().plot(x='Sample Datetime', y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='scatter', rot=45, label = 'First Bottle')
#result_05212025.result_df[~first_rep].loc['Dobson'].reset_index().plot(x='Sample Datetime', y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='scatter', rot=45,color='r', label = 'Second Bottle', title = 'Dobson Nutrient Analyzer Data from 5-21-25 Run')

- yikes. Dobson data is straight trash. huge variability between bottle replicates.
- I suspect this has to less to do with it being Dobson or RME and more to do with it being samples I filtered vs samples josh filtered. Or could be that the RME samples were filtered very freshly.

# 6-10-25 Nutrient Analyzer Run QAQC Run 1

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250610 Run/'

result_061025_1 = AA500.AA500_Result(result_dir + 'RCEW_NOx_PO4_NH3_11Jun25_R1.TXT', result_dir+'samplelist_20250610_1.xlsx')

result_061025_1.result_df

## Drift and Baseline

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_061025_1.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_061025_1.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_061025_1.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- nitrate ran out of reagent after 8th drift standard - need to rerun cups 91 on.
- ammonium drifted after 10th drift standard

## Blank Check

### Instrument Blank

In [ ]:
result_061025_1.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- all nitrate blanks look good. 
- phosphate and ammonium blanks comparable to 5-21-25, but lacking context in expected phosphate and ammonium levels

In [ ]:
result_061025_1.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

### Reagent blank

In [ ]:
result_061025_1.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

In [ ]:
result_061025_1.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

- no nitrate contamination from reagents
- third reagent blank has some phosphate and ammonium contamination from the sodium carbonate it looks like - could be from tip into sample

### Field Blank


In [ ]:
result_061025_1.result_df.loc['Field Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

In [ ]:
result_061025_1.result_df.loc['Field Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

- no contamination from field blanks

## AA500 In-Sample Variation

In [ ]:
result_061025_1.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of .9 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples ok, but a few bad ones - throw those out
- qa threshold of 5 ug/l

phosphate:
- ok - mean of 6 ug/L greater than 4 ug/L in SEAL datasheet
- also a bad one or two - max of 38 ug/L!
- qa threshold of 4 ug/l

ammonium:
- way better than seal spec! 75% less than 1.5 ug/L, seal spec is 2 ug/L
- one bad one with a max of 8 ug/L
- qa threshold of 2 ug/l


In [ ]:
result_061025_1.result_df.loc['RME']

In [ ]:
result_061025_1.result_df.loc['Dobson']

- nice. for nitrate QA flag, RME has only a few
- most flags for phosphate, a couple for ammonium. but no wholesale bad data
- several really high phosphate readings (~.2 mg/L) that get flagged with a relatively low std (<8 ug/L). could up the threshold

## Between Bottle Variation

In [ ]:
plot_between_bottle_bar(result_061025_1.result_df.loc['RME'], 'Nitrate')


In [ ]:
plot_between_bottle_bar(result_061025_1.result_df.loc['Dobson'], 'Nitrate')


- not perfect, but better than than 5-21 run. are these my or josh's filtrates that are good?

### Is who filtered the samples responsible for the variation?

In [ ]:
ax= plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='JCS'], 'Nitrate', title= 'Josh')


In [ ]:
ax = plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='KF'], 'Nitrate', title= 'Kyle')

In [ ]:
ax = plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='BE'], 'Nitrate', title= 'Ben')

- yes, Josh's show more variation. kyle has a bad one, and mine aren't perfect though

In [ ]:
result_061025_1.result_df.loc['RME']

In [ ]:
result_061025_1.result_df.loc['Dobson']

- RME has some BBV and ISV flags, but not many. Dobson has far more. most RME data look usable
- remember to only plot data from sites that are of type Hand, Autosampler, or Manual

# Joining All Nutrient Analyzer Runs

In [ ]:
class ReynoldsResult(pd.DataFrame):
    def get_all_params(self):
        return [col.split()[0] for col in df.columns if col.endswith(' QA')]
    
    def get_valid_data(self):
        params = self.get_all_params()
        for param in params:
            valid = self[param + ' QA'] == ''
            print(param)
            self.loc[~valid, [param + ' mean', param + ' err', param + ' std']] = np.NaN
        return self

    def get_sample_data(self):
        sample = (self['Sample Type'] == 'Autosampler') | (self['Sample Type'] == 'Hand') | (self['Sample Type'] == 'Manual')
        return self.loc[sample]

In [ ]:
nutrient_results = ReynoldsResult(pd.concat([result_05212025.result_df, result_061025_1.result_df]).sort_index())
nutrient_results

In [ ]:
nutrient_results.get_valid_data().loc['RME']

# Plotting Nutrient Analyzer Data with SCAN Data

## Timeseries

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)
rme_cleaned

In [ ]:
def plot_nutrient_data(df, start_date, end_date, site, param, ax, check_valid = True):
    first_rep = df['Bottle Replicate'] == 1

    df[first_rep].loc[site].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = 'First Bottle')
    df[~first_rep].loc[site].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', color = 'r', rot=45, label = 'Second Bottle')


In [ ]:
nutrient_results.get_valid_data().index

In [ ]:

fig, ax= plt.subplots(figsize=(11,8.5))



rme_cleaned.loc['2-13-25':'4-02-25'].plot(y= ['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax, rot=45)
plot_nutrient_data(nutrient_results.get_valid_data().get_sample_data().dropna(subset='Nitrate mean'), '2/13/25', '4/2/25', 'RME', 'Nitrate', ax)


- well cool. there's a couple outliers, on 2/15/25 3:30:00 and another 2-22-25 11:00. but overall theses data catch the general form of the S::CAN data. we can work with it.
- best fit looks to be two_wavelength_no3_mgl_correct, seems to fit better than uncorrected version, which is second best. 
- one_wavelength corrected not plotted, its way off

In [ ]:
fig, ax= plt.subplots(figsize=(11,8.5))

rme_cleaned.loc['2-13-25':'4-03-25'].plot(y= ['two_wavelength_no3_mgl_correct'], ax=ax, rot=45)
plot_nutrient_data(nutrient_results.get_valid_data().get_sample_data().dropna(subset='Nitrate mean'), '2/13/25', '4/2/25', 'RME', 'Nitrate', ax)


## Correlation Plots

In [ ]:
rme_aa500 =  nutrient_results.get_valid_data().get_sample_data().dropna(subset='Nitrate mean').loc['RME']

rme_merged = pd.merge_asof(rme_aa500, rme_cleaned, left_index=True, right_index=True, direction='nearest')


In [ ]:
rme_merged_clean = rme_merged.drop(['2/15/25 3:30:00', '2-22-25 11:00']).dropna(subset='Nitrate mean') # drop two outliers from timeseries

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(11,8.5))

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .2, .02, ax[0,0])

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, 2, .3, ax[1,0])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[0,1])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,2], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[0,2])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,2], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[1,2])


fig.suptitle('RME Calibration Plots')
fig.tight_layout()
